# 🏥 Disease & Symptoms — Interactive Analytics Dashboard
### Python Notebook | Mirrors a Full Power BI Dashboard
---
**Dataset:** 773 Diseases · 377 Symptoms · ~246,000 rows  
**What this covers:** Data loading → Cleaning → Power Query-style transforms → DAX-style measures → Interactive visuals → Full dashboard layout


## ⚙️ Step 0 — Install & Import Dependencies

In [ ]:
# Install required libraries (run once)
# !pip install kagglehub[pandas-datasets] plotly pandas numpy

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Set default theme (mirrors Power BI professional look)
pio.templates.default = "plotly_white"

# Colour palette (Power BI inspired)
BLUE       = "#1E3A5F"
ACCENT     = "#2E75B6"
ORANGE     = "#ED7D31"
GREEN      = "#70AD47"
RED        = "#C00000"
LIGHT_BLUE = "#BDD7EE"
GOLD       = "#FFC000"

SEVERITY_COLOURS = {
    "Critical": RED,
    "Moderate": ORANGE,
    "Mild":     GREEN
}

print("✅ All libraries imported successfully.")


## 📥 Step 1 — Load Dataset (via KaggleHub)

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

print("⏳ Downloading dataset from Kaggle...")

df_raw = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "dhivyeshrk/diseases-and-symptoms-dataset",
    ""
)

print(f"✅ Dataset loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print("\nColumn names:")
print(df_raw.columns.tolist())
df_raw.head(3)


## 🔍 Step 2 — Initial Data Exploration

In [ ]:
print("=" * 55)
print("DATASET SHAPE")
print(f"  Rows    : {df_raw.shape[0]:,}")
print(f"  Columns : {df_raw.shape[1]}")
print("=" * 55)
print("\nDATATYPES")
print(df_raw.dtypes)
print("\nNULL COUNTS")
print(df_raw.isnull().sum())
print("\nSAMPLE ROWS")
df_raw.head(5)


## 🔄 Step 3 — Power Query–Style Transformations
> Equivalent to: Get Data → Transform Tab → Unpivot → Merge → Conditional Column in Power BI


In [ ]:
df = df_raw.copy()

# ── 3.1 Standardise column names ──────────────────────────
df.columns = df.columns.str.strip().str.replace(' ', '_')

# Identify disease column and symptom columns
disease_col = df.columns[0]          # first column = Disease
symptom_cols = [c for c in df.columns if 'symptom' in c.lower() or 'Symptom' in c]

print(f"Disease column  : {disease_col}")
print(f"Symptom columns : {symptom_cols[:5]} ... ({len(symptom_cols)} total)")

# ── 3.2 Trim & clean text (Power Query: Transform > Format > Trim) ─
df[disease_col] = df[disease_col].astype(str).str.strip().str.title()
for col in symptom_cols:
    df[col] = df[col].astype(str).str.strip().str.lower().replace({'nan': None, '': None})

print("\n✅ Text cleaned and trimmed.")


In [ ]:
# ── 3.3 UNPIVOT symptom columns (Power Query: Unpivot Other Columns) ─
# This transforms wide format → tall format (one row per symptom)

id_vars = [disease_col]   # keep Disease as anchor

df_unpivoted = df.melt(
    id_vars     = id_vars,
    value_vars  = symptom_cols,
    var_name    = "Symptom_Position",   # was Symptom_1, Symptom_2 ...
    value_name  = "Symptom"
)

# Drop blank/null symptoms (Power Query: Filter > Remove Blanks)
df_long = df_unpivoted.dropna(subset=["Symptom"]).copy()
df_long = df_long[df_long["Symptom"].str.strip() != ""]
df_long.reset_index(drop=True, inplace=True)

print(f"After unpivot & clean:")
print(f"  Rows    : {df_long.shape[0]:,}")
print(f"  Columns : {df_long.columns.tolist()}")
df_long.head(5)


In [ ]:
# ── 3.4 Build a Severity lookup (Symptom → Score) ─────────
# Simulated Symptom-severity.csv mapping (if you have the actual file, load it instead)
# Creates a numeric severity weight per unique symptom

unique_symptoms = df_long["Symptom"].unique()
np.random.seed(42)
severity_map = {s: int(np.random.choice(range(1, 8), p=[0.1,0.15,0.2,0.2,0.15,0.1,0.1]))
                for s in unique_symptoms}

df_long["SeverityScore"] = df_long["Symptom"].map(severity_map)

# ── 3.5 Conditional Column (Power BI: Add Column > Conditional Column) ─
def severity_category(score):
    if pd.isna(score):   return "Unknown"
    if score >= 6:       return "Critical"
    if score >= 4:       return "Moderate"
    return "Mild"

df_long["SeverityCategory"] = df_long["SeverityScore"].apply(severity_category)

print("SeverityCategory distribution:")
print(df_long["SeverityCategory"].value_counts())
df_long.head(3)


## 🗂️ Step 4 — Build Dimension Tables (Star Schema)
> Equivalent to Power BI Model View: Fact Table + Dimension Tables


In [ ]:
# ── FACT TABLE ─────────────────────────────────────────────
fact = df_long.copy()
fact.rename(columns={disease_col: "Disease"}, inplace=True)
fact["DiseaseID"] = fact["Disease"].astype("category").cat.codes + 1
fact["SymptomID"]  = fact["Symptom"].astype("category").cat.codes + 1

# ── DIMENSION: Disease ─────────────────────────────────────
dim_disease = (fact[["DiseaseID","Disease"]]
               .drop_duplicates()
               .sort_values("DiseaseID")
               .reset_index(drop=True))

# ── DIMENSION: Symptom ─────────────────────────────────────
dim_symptom = (fact[["SymptomID","Symptom","SeverityScore","SeverityCategory"]]
               .drop_duplicates(subset=["SymptomID"])
               .sort_values("SymptomID")
               .reset_index(drop=True))

print(f"Fact table       : {fact.shape[0]:,} rows")
print(f"Dim Disease      : {dim_disease.shape[0]:,} unique diseases")
print(f"Dim Symptom      : {dim_symptom.shape[0]:,} unique symptoms")


## 📐 Step 5 — DAX–Style Measures (Python Functions)
> Equivalent to: New Measure → CALCULATE, RANKX, DIVIDE, DISTINCTCOUNT in Power BI


In [ ]:
# ── MEASURE: Total Records ─────────────────────────────────
total_records = len(fact)

# ── MEASURE: Unique Diseases ───────────────────────────────
unique_diseases = fact["Disease"].nunique()

# ── MEASURE: Unique Symptoms ───────────────────────────────
unique_symptoms = fact["Symptom"].nunique()

# ── MEASURE: Avg Severity ──────────────────────────────────
avg_severity = round(fact["SeverityScore"].mean(), 2)

# ── MEASURE: Disease Count (rows per disease = occurrence probability) ─
disease_count = (fact.groupby("Disease")
                     .size()
                     .reset_index(name="DiseaseCount")
                     .sort_values("DiseaseCount", ascending=False))

# ── MEASURE: Disease Rank (RANKX equivalent) ──────────────
disease_count["Rank"] = disease_count["DiseaseCount"].rank(
    method="dense", ascending=False).astype(int)

# ── MEASURE: % of Total (DIVIDE + ALL equivalent) ─────────
disease_count["PctOfTotal"] = (
    disease_count["DiseaseCount"] / total_records * 100).round(2)

# ── MEASURE: Symptom Count per Disease ────────────────────
symptom_per_disease = (fact.groupby("Disease")["Symptom"]
                           .nunique()
                           .reset_index(name="UniqueSymptoms"))

# ── MEASURE: Avg Severity per Disease ─────────────────────
avg_sev_disease = (fact.groupby("Disease")["SeverityScore"]
                       .mean()
                       .reset_index(name="AvgSeverity")
                       .round({"AvgSeverity": 2}))

print("📊 KPI Summary")
print(f"  Total Records   : {total_records:,}")
print(f"  Unique Diseases : {unique_diseases:,}")
print(f"  Unique Symptoms : {unique_symptoms:,}")
print(f"  Avg Severity    : {avg_severity}")
print("\nTop 5 Diseases by Count:")
print(disease_count.head())


## 📊 Step 6 — Visual 1: KPI Cards
> Equivalent to: Card Visual in Power BI (top row of the dashboard)


In [ ]:
fig_kpi = make_subplots(
    rows=1, cols=4,
    subplot_titles=["Total Records", "Unique Diseases", "Unique Symptoms", "Avg Severity Score"]
)

kpi_values = [
    (f"{total_records:,}", ACCENT),
    (f"{unique_diseases:,}", BLUE),
    (f"{unique_symptoms:,}", GREEN),
    (f"{avg_severity}", ORANGE),
]

for i, (val, colour) in enumerate(kpi_values, 1):
    fig_kpi.add_trace(
        go.Indicator(
            mode="number",
            value=float(val.replace(",","")),
            number={"font": {"size": 48, "color": colour, "family": "Segoe UI"},
                    "valueformat": ","},
        ),
        row=1, col=i
    )

fig_kpi.update_layout(
    height=200,
    paper_bgcolor="white",
    margin=dict(l=20, r=20, t=60, b=10),
    title_text="",
)
fig_kpi.show()


## 📊 Step 7 — Visual 2: Top 10 Diseases by Occurrence
> Equivalent to: Clustered Bar Chart + Top N Filter + RANKX in Power BI


In [ ]:
top10 = disease_count[disease_count["Rank"] <= 10].sort_values("DiseaseCount")

fig_bar = go.Figure(go.Bar(
    x=top10["DiseaseCount"],
    y=top10["Disease"],
    orientation="h",
    marker_color=ACCENT,
    text=top10["DiseaseCount"].apply(lambda x: f"{x:,}"),
    textposition="outside",
    customdata=top10[["PctOfTotal", "Rank"]],
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Count: %{x:,}<br>"
        "% of Total: %{customdata[0]:.2f}%<br>"
        "Rank: #%{customdata[1]}<extra></extra>"
    )
))

fig_bar.update_layout(
    title=dict(text="🏆 Top 10 Most Common Diseases", font=dict(size=18, color=BLUE)),
    xaxis_title="Number of Records",
    yaxis_title="",
    height=450,
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="#E0E0E0"),
    yaxis=dict(autorange=True),
    margin=dict(l=180, r=80, t=60, b=40),
)
fig_bar.show()


## 📊 Step 8 — Visual 3: Severity Category Distribution
> Equivalent to: Donut Chart with conditional colour formatting in Power BI


In [ ]:
sev_dist = fact["SeverityCategory"].value_counts().reset_index()
sev_dist.columns = ["SeverityCategory", "Count"]

fig_donut = go.Figure(go.Pie(
    labels=sev_dist["SeverityCategory"],
    values=sev_dist["Count"],
    hole=0.55,
    marker_colors=[SEVERITY_COLOURS.get(s, "#888") for s in sev_dist["SeverityCategory"]],
    textinfo="label+percent",
    textfont=dict(size=13),
    hovertemplate="<b>%{label}</b><br>Count: %{value:,}<br>Share: %{percent}<extra></extra>",
))

fig_donut.update_layout(
    title=dict(text="🩺 Cases by Severity Category", font=dict(size=18, color=BLUE)),
    annotations=[dict(text="Severity", x=0.5, y=0.5, font_size=16,
                      font_color=BLUE, showarrow=False)],
    height=420,
    paper_bgcolor="white",
    legend=dict(orientation="v", x=1.02),
    margin=dict(l=20, r=120, t=60, b=20),
)
fig_donut.show()


## 📊 Step 9 — Visual 4: Top 20 Most Frequent Symptoms
> Equivalent to: Bar Chart with Top N filter on Symptom dimension


In [ ]:
symptom_count = (fact.groupby("Symptom")
                     .agg(Count=("Disease","count"),
                          AvgSeverity=("SeverityScore","mean"),
                          Diseases=("Disease","nunique"))
                     .reset_index()
                     .sort_values("Count", ascending=False)
                     .head(20))

symptom_count["AvgSeverity"] = symptom_count["AvgSeverity"].round(2)

fig_sym = go.Figure(go.Bar(
    x=symptom_count["Count"],
    y=symptom_count["Symptom"],
    orientation="h",
    marker=dict(
        color=symptom_count["AvgSeverity"],
        colorscale=[[0,"#70AD47"],[0.5,"#FFC000"],[1,"#C00000"]],
        colorbar=dict(title="Avg Severity"),
        showscale=True,
    ),
    text=symptom_count["Count"].apply(lambda x: f"{x:,}"),
    textposition="outside",
    customdata=symptom_count[["AvgSeverity","Diseases"]],
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Occurrences: %{x:,}<br>"
        "Avg Severity: %{customdata[0]}<br>"
        "Distinct Diseases: %{customdata[1]}<extra></extra>"
    )
))

fig_sym.update_layout(
    title=dict(text="💊 Top 20 Most Frequent Symptoms (coloured by Severity)", font=dict(size=18, color=BLUE)),
    height=580,
    xaxis_title="Occurrences",
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="#E0E0E0"),
    margin=dict(l=200, r=100, t=60, b=40),
)
fig_sym.show()


## 📊 Step 10 — Visual 5: Heatmap Matrix (Disease × Symptom Position)
> Equivalent to: Matrix visual with Conditional Formatting (heatmap) in Power BI


In [ ]:
# Focus on top 15 diseases for readability
top15_diseases = disease_count.head(15)["Disease"].tolist()

matrix_df = (fact[fact["Disease"].isin(top15_diseases)]
             .groupby(["Disease","Symptom_Position"])
             .size()
             .reset_index(name="Count"))

# Pivot to wide format
matrix_pivot = matrix_df.pivot(
    index="Disease", columns="Symptom_Position", values="Count"
).fillna(0)

# Keep first 10 symptom positions for clean display
cols_to_show = sorted(matrix_pivot.columns)[:10]
matrix_pivot = matrix_pivot[cols_to_show]

fig_heat = go.Figure(go.Heatmap(
    z=matrix_pivot.values,
    x=matrix_pivot.columns.tolist(),
    y=matrix_pivot.index.tolist(),
    colorscale=[[0,"#FFFFFF"],[0.3,LIGHT_BLUE],[0.7,ACCENT],[1,BLUE]],
    text=matrix_pivot.values.astype(int),
    texttemplate="%{text}",
    textfont=dict(size=11),
    hovertemplate="<b>Disease:</b> %{y}<br><b>Position:</b> %{x}<br><b>Count:</b> %{z}<extra></extra>",
    colorbar=dict(title="Count"),
))

fig_heat.update_layout(
    title=dict(text="🔥 Symptom Frequency Heatmap — Top 15 Diseases × Symptom Position",
               font=dict(size=17, color=BLUE)),
    height=500,
    xaxis_title="Symptom Position",
    yaxis_title="",
    paper_bgcolor="white",
    margin=dict(l=200, r=80, t=70, b=60),
)
fig_heat.show()


## 📊 Step 11 — Visual 6: Disease Risk Scatter Plot
> Equivalent to: Scatter Chart with Play Axis / Bubble Chart in Power BI


In [ ]:
# Merge measures for each disease
scatter_df = (disease_count[["Disease","DiseaseCount","PctOfTotal","Rank"]]
              .merge(avg_sev_disease, on="Disease")
              .merge(symptom_per_disease, on="Disease"))

# Colour by avg severity level
scatter_df["SeverityLevel"] = pd.cut(
    scatter_df["AvgSeverity"],
    bins=[0, 3, 5, 8],
    labels=["Mild", "Moderate", "Critical"]
)

fig_scatter = px.scatter(
    scatter_df,
    x="AvgSeverity",
    y="DiseaseCount",
    size="UniqueSymptoms",
    color="SeverityLevel",
    color_discrete_map=SEVERITY_COLOURS,
    hover_name="Disease",
    hover_data={"DiseaseCount": True, "AvgSeverity": True,
                "UniqueSymptoms": True, "PctOfTotal": True},
    labels={"DiseaseCount": "Occurrence Count",
            "AvgSeverity":  "Average Severity Score",
            "UniqueSymptoms": "Unique Symptoms"},
    title="⚠️ Disease Risk Matrix — Frequency vs Severity (bubble = symptom diversity)",
)
fig_scatter.update_layout(
    height=520,
    paper_bgcolor="white",
    plot_bgcolor="white",
    title_font=dict(size=17, color=BLUE),
    xaxis=dict(showgrid=True, gridcolor="#E0E0E0"),
    yaxis=dict(showgrid=True, gridcolor="#E0E0E0"),
)
fig_scatter.show()


## 📊 Step 12 — Visual 7: Treemap — Disease Share
> Equivalent to: Treemap visual in Power BI


In [ ]:
top30 = disease_count.head(30).merge(avg_sev_disease, on="Disease")

fig_tree = px.treemap(
    top30,
    path=["Disease"],
    values="DiseaseCount",
    color="AvgSeverity",
    color_continuous_scale=[[0,"#70AD47"],[0.5,"#FFC000"],[1,"#C00000"]],
    hover_data={"PctOfTotal": True, "Rank": True},
    title="🗺️ Top 30 Diseases — Treemap (size = occurrence, colour = severity)",
    color_continuous_midpoint=4
)
fig_tree.update_layout(
    height=520,
    paper_bgcolor="white",
    title_font=dict(size=17, color=BLUE),
    coloraxis_colorbar=dict(title="Avg<br>Severity"),
    margin=dict(t=60, l=10, r=10, b=10),
)
fig_tree.show()


## 🔎 Step 13 — Drill-Through: Disease Detail View
> Equivalent to: Right-click → Drill Through page in Power BI
> Change `selected_disease` to explore any disease interactively


In [ ]:
# ── CHANGE THIS TO ANY DISEASE YOU WANT TO DRILL INTO ─────
selected_disease = disease_count["Disease"].iloc[0]   # auto-picks #1 most common

print(f"🔍 Drilling through: {selected_disease}")
print("─" * 50)

drill = fact[fact["Disease"] == selected_disease].copy()

# KPI for this disease
d_count    = len(drill)
d_symptoms = drill["Symptom"].nunique()
d_avg_sev  = round(drill["SeverityScore"].mean(), 2)
d_pct      = round(d_count / total_records * 100, 3)

print(f"  Records        : {d_count:,}")
print(f"  Unique Symptoms: {d_symptoms}")
print(f"  Avg Severity   : {d_avg_sev}")
print(f"  % of Total Data: {d_pct}%")

# Build sub-dashboard for this disease
fig_drill = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f"Symptom Frequency — {selected_disease}",
        f"Severity Breakdown — {selected_disease}"
    ]
)

# Left: symptom bar
sym_drill = (drill.groupby("Symptom")
                  .size()
                  .reset_index(name="Count")
                  .sort_values("Count", ascending=True)
                  .tail(15))

fig_drill.add_trace(
    go.Bar(x=sym_drill["Count"], y=sym_drill["Symptom"],
           orientation="h", marker_color=ACCENT,
           name="Symptom Count"),
    row=1, col=1
)

# Right: severity donut
sev_drill = drill["SeverityCategory"].value_counts().reset_index()
sev_drill.columns = ["Cat","Cnt"]

fig_drill.add_trace(
    go.Pie(labels=sev_drill["Cat"], values=sev_drill["Cnt"], hole=0.5,
           marker_colors=[SEVERITY_COLOURS.get(s,"#888") for s in sev_drill["Cat"]],
           name="Severity"),
    row=1, col=2
)

fig_drill.update_layout(
    height=420,
    title_text=f"📋 Drill-Through Detail: {selected_disease}",
    title_font=dict(size=17, color=BLUE),
    paper_bgcolor="white",
    showlegend=False,
)
fig_drill.show()


## 🎚️ Step 14 — Slicer Simulation (Interactive Filter)
> Equivalent to: Slicer visual in Power BI — filter all charts by Severity or Symptom


In [ ]:
# ── CHANGE THESE VALUES TO SIMULATE SLICER SELECTIONS ─────
FILTER_SEVERITY = "Critical"      # Options: "Critical", "Moderate", "Mild", None
FILTER_SYMPTOM  = None            # e.g. "fatigue", or None for all symptoms
# ──────────────────────────────────────────────────────────

filtered = fact.copy()
if FILTER_SEVERITY:
    filtered = filtered[filtered["SeverityCategory"] == FILTER_SEVERITY]
if FILTER_SYMPTOM:
    filtered = filtered[filtered["Symptom"] == FILTER_SYMPTOM.lower()]

print(f"Active Filters → Severity: {FILTER_SEVERITY or 'All'}  |  Symptom: {FILTER_SYMPTOM or 'All'}")
print(f"Filtered rows  : {len(filtered):,} / {total_records:,}  ({len(filtered)/total_records*100:.1f}%)")

# Filtered top 10 diseases
f_top10 = (filtered.groupby("Disease")
                   .size()
                   .reset_index(name="Count")
                   .sort_values("Count", ascending=False)
                   .head(10)
                   .sort_values("Count"))

fig_f = go.Figure(go.Bar(
    x=f_top10["Count"],
    y=f_top10["Disease"],
    orientation="h",
    marker_color=RED if FILTER_SEVERITY == "Critical" else (ORANGE if FILTER_SEVERITY == "Moderate" else GREEN),
    text=f_top10["Count"].apply(lambda x: f"{x:,}"),
    textposition="outside",
))
fig_f.update_layout(
    title=dict(
        text=f"Top 10 Diseases — Filtered by: {FILTER_SEVERITY or 'All'} severity",
        font=dict(size=17, color=BLUE)
    ),
    height=420,
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="#E0E0E0"),
    margin=dict(l=180, r=80, t=60, b=40),
)
fig_f.show()


## 🖥️ Step 15 — Full Combined Dashboard (Single View)
> Equivalent to: Full Power BI report page with all visuals in one layout


In [ ]:
fig_dash = make_subplots(
    rows=3, cols=3,
    specs=[
        [{"type":"indicator"}, {"type":"indicator"}, {"type":"indicator"}],
        [{"type":"xy", "colspan":2}, None, {"type":"domain"}],
        [{"type":"xy"}, {"type":"xy"}, {"type":"domain"}],
    ],
    subplot_titles=[
        "Total Records", "Unique Diseases", "Avg Severity",
        "Top 10 Diseases", "", "Severity Split",
        "Top 15 Symptoms", "Disease Frequency Trend", "Symptom Position Mix"
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.06,
)

# ── Row 1: KPI Cards ────────────────────────────────────────
for col, (val, colour, label) in enumerate([
    (total_records,   ACCENT, ""),
    (unique_diseases, BLUE,   ""),
    (avg_severity,    ORANGE, ""),
], 1):
    fig_dash.add_trace(
        go.Indicator(mode="number",
                     value=float(val),
                     number={"font":{"size":40,"color":colour}}),
        row=1, col=col
    )

# ── Row 2 Left: Top 10 Diseases bar ─────────────────────────
t10 = disease_count.head(10).sort_values("DiseaseCount")
fig_dash.add_trace(
    go.Bar(x=t10["DiseaseCount"], y=t10["Disease"],
           orientation="h", marker_color=ACCENT,
           name="Disease Count", showlegend=False),
    row=2, col=1
)

# ── Row 2 Right: Severity donut ─────────────────────────────
fig_dash.add_trace(
    go.Pie(labels=sev_dist["SeverityCategory"], values=sev_dist["Count"],
           hole=0.55, showlegend=True,
           marker_colors=[SEVERITY_COLOURS.get(s,"#888") for s in sev_dist["SeverityCategory"]]),
    row=2, col=3
)

# ── Row 3 Left: Top 15 Symptoms bar ─────────────────────────
s15 = (fact.groupby("Symptom").size()
           .reset_index(name="Cnt")
           .sort_values("Cnt", ascending=False)
           .head(15).sort_values("Cnt"))
fig_dash.add_trace(
    go.Bar(x=s15["Cnt"], y=s15["Symptom"],
           orientation="h", marker_color=GREEN,
           name="Symptom Count", showlegend=False),
    row=3, col=1
)

# ── Row 3 Mid: Disease count line ───────────────────────────
top20_line = disease_count.head(20)
fig_dash.add_trace(
    go.Scatter(x=list(range(1, 21)), y=top20_line["DiseaseCount"],
               mode="lines+markers", line_color=ACCENT,
               name="Disease Count", showlegend=False),
    row=3, col=2
)

# ── Row 3 Right: Symptom Position distribution ──────────────
pos_dist = (fact.groupby("Symptom_Position")
                .size()
                .reset_index(name="Count")
                .sort_values("Count", ascending=False)
                .head(8))
fig_dash.add_trace(
    go.Pie(labels=pos_dist["Symptom_Position"],
           values=pos_dist["Count"],
           hole=0.4, showlegend=False,
           textinfo="label+percent"),
    row=3, col=3
)

fig_dash.update_layout(
    height=950,
    title_text="🏥 Disease & Symptoms — Full Analytics Dashboard",
    title_font=dict(size=22, color=BLUE, family="Segoe UI"),
    paper_bgcolor="#F8FAFD",
    plot_bgcolor="white",
    font=dict(family="Segoe UI", size=11),
    margin=dict(l=30, r=30, t=80, b=30),
    legend=dict(orientation="h", x=0.7, y=0.55),
)
fig_dash.show()
print("\n✅ Full dashboard rendered. Save with:  fig_dash.write_html('disease_dashboard.html')")


## 💾 Step 16 — Export Dashboard to HTML
> Save the full interactive dashboard as a standalone HTML file (shareable, no Power BI needed)


In [ ]:
# Save all individual charts
fig_bar.write_html("chart_top10_diseases.html")
fig_donut.write_html("chart_severity_donut.html")
fig_sym.write_html("chart_top20_symptoms.html")
fig_heat.write_html("chart_heatmap_matrix.html")
fig_scatter.write_html("chart_risk_scatter.html")
fig_tree.write_html("chart_treemap.html")
fig_dash.write_html("disease_dashboard_FULL.html")

print("✅ All charts exported as interactive HTML files!")
print("\nFiles created:")
files = [
    "disease_dashboard_FULL.html  ← Open this for the complete dashboard",
    "chart_top10_diseases.html",
    "chart_severity_donut.html",
    "chart_top20_symptoms.html",
    "chart_heatmap_matrix.html",
    "chart_risk_scatter.html",
    "chart_treemap.html",
]
for f in files:
    print(f"  📄 {f}")


---
## ✅ Project Completion Summary

| Power BI Feature | Python Equivalent Used |
|---|---|
| Get Data / Power Query | `kagglehub.load_dataset` + `pandas` |
| Transform Tab (Trim, Type) | `str.strip()`, `astype()` |
| Unpivot Columns | `pd.melt()` |
| Merge Queries | `pd.merge()` |
| Conditional Column | `pd.cut()` / custom function |
| Star Schema (Model View) | Separate `fact`, `dim_disease`, `dim_symptom` DataFrames |
| Measures (COUNTROWS) | `len(fact)` |
| DISTINCTCOUNT | `.nunique()` |
| CALCULATE / ALL | `groupby + agg` with full scope |
| RANKX | `.rank()` |
| DIVIDE (% of Total) | `value / total * 100` |
| Card Visual | `go.Indicator` |
| Bar Chart + Top N Filter | `go.Bar` + `.head(10)` |
| Donut Chart | `go.Pie(hole=0.55)` |
| Matrix + Conditional Format | `go.Heatmap` |
| Scatter / Bubble Chart | `px.scatter(size=...)` |
| Treemap | `px.treemap` |
| Drill-Through Page | Filtered sub-dashboard function |
| Slicers (Filters) | Python variable filters |
| Full Dashboard Layout | `make_subplots` |
| Publish / Share | `.write_html()` → shareable file |

> **All charts are interactive** — hover, zoom, click legend items to filter, and pan. 🎉
